# Import Environment variables

In [1]:
%run /home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/Setting_Env_Variables.ipynb

Found bucket: id=rw-migration-aou-rw-f7a4d148, bucketName=rw-migration-aou-rw-f7a4d148
-> Assigned migration variables (ID: rw-migration-aou-rw-f7a4d148)
Found bucket: id=temporary-workspace-bucket, bucketName=temporary-workspace-bucket-wb-perky-cabbage-8342
Found bucket: id=workspace-bucket, bucketName=workspace-bucket-wb-perky-cabbage-8342
✅ Successfully identified latest dataset: wb-silky-artichoke-2408.C2024Q3R9

Variables extracted:
GOOGLE_CLOUD_PROJECT: wb-perky-cabbage-8342
WORKSPACE_BUCKET: gs://workspace-bucket-wb-perky-cabbage-8342
WORKSPACE_TEMP_BUCKET: gs://temporary-workspace-bucket-wb-perky-cabbage-8342
WORKSPACE_CDR: wb-silky-artichoke-2408.C2024Q3R9
bucket_aou_tutorial: NOT FOUND
bucket_id_aou_tutorial: NOT FOUND
bucket_migrated: gs://rw-migration-aou-rw-f7a4d148
bucket_id_migrated: rw-migration-aou-rw-f7a4d148

✅ Saved to /home/jupyter/.bashrc
C2024Q3R9 BQ_DATASET
Multi-trait-GWAS-in-admixed-populations GIT_REPO
dataset_test2 BQ_DATASET
prep_C2024Q3R9 BQ_DATASET
rw-mig

In [2]:
%run /home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/Setting_Env_Variables_p2.ipynb

WORKSPACE_CDR = wb-silky-artichoke-2408.C2024Q3R9
WORKSPACE_BUCKET = gs://workspace-bucket-wb-perky-cabbage-8342
GOOGLE_PROJECT = wb-perky-cabbage-8342
Done! 10 variables saved to: /home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/Setting_Env_Variables_p2.R
Done! 10 variables saved to: /home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/Setting_Env_Variables.sas


# Librairy

In [3]:
import os
import numpy as np
import pandas as pd
from google.cloud import bigquery

# Data's import

## Clinical data

In [4]:
# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

name_of_file_in_bucket = "df_bc_ko_at_inclusion_223k_nb_BC_WITHIN_5Y_GeneticAncestry_biopsy_FamilyHistory_LastSelection.tsv"

df = pd.read_csv(my_bucket +'/Data/'+ name_of_file_in_bucket, sep=',', low_memory=False)

In [5]:
df.drop(columns=['gender','date_of_birth','ethnicity','self_reported_category','time_category','delay_years',
                 'first_any_biopsy_date','days_inclusion_to_biopsy', 'days_biopsy_to_cancer',
                 'first_survey_date', 'last_survey_date','days_last_survey_to_first_survey', 'days_inclusion_to_first_survey',
                 'days_inclusion_to_last_survey', 'days_has_bc_to_first_survey','days_has_bc_to_last_survey',
                 'date_1_of_biopsy_19081_19086', 'date_2_of_biopsy_19081_19086', 'date_3_of_biopsy_19081_19086',
                 'has_biopsy_19081_19086',
                 'date_1_of_biopsy_19100_19103', 'date_2_of_biopsy_19100_19103', 'date_3_of_biopsy_19100_19103',
                 'has_biopsy_19100_19103',
                 'date_1_of_biopsy_19120_19125_19126', 'date_2_of_biopsy_19120_19125_19126', 'date_3_of_biopsy_19120_19125_19126',
                 'has_biopsy_19120_19125_19126',
                 'Daughter', 'Mother', 'Sibling',
                 'statut_parente'], inplace=True)

In [6]:
df.columns

Index(['person_id', 'inclusion_date', 'first_breast_cancer_date', 'delay_days',
       'age_at_inclusion', 'has_bc', 'eur_rye', 'eas_rye', 'amr_rye',
       'afr_rye', 'sas_rye', 'mid_rye', 'dominant_origin', 'ancestry_80',
       'biopsy_result', 'age_category', 'race_us_bcsc', 'race_us_bcsc_detail',
       'breast_cancer_family_history_first_degree'],
      dtype='object')

In [7]:
my_indivs = list(df['person_id'])

# Contextual informations

__Sept facteurs de risque de cancer du sein ont été pris en compte :__

* l’âge des premières règles ;
* la parité (antécédents d’accouchement) ;
* l’âge à la première grossesse à terme ;
* l’indice de masse corporelle (IMC) à l’âge adulte chez les femmes ménopausées ;
* la taille à l’âge adulte ;
* le recours actuel à un traitement hormonal de la ménopause (THM) à base d’œstrogènes et de progestérone ;
* la consommation moyenne d’alcool au cours de la vie.

__Harmonisation des données et définitions des variables__

* _Les variables dépendantes du temps ont été évaluées à une date de référence définie comme la date du diagnostic pour les cas et la date de l'entretien pour les témoins dans les études cas-témoins. [..] la date de référence était celle du dernier questionnaire de suivi, si disponible ; sinon, la date du questionnaire initial a été utilisée._

* _En l'absence de données sur le statut ménopausique, nous avons utilisé l'âge médian (54 ans) comme indicateur de substitution : les femmes âgées de moins de 54 ans ont été considérées comme préménopausiques et celles âgées de 54 ans ou plus comme postménopausiques._

* _Le recours actuel à un THM à base d’œstrogènes et de progestérone a été défini comme un recours dans les six mois précédant la date de référence._

* _Dans les études cas-témoins, l’IMC a été calculé à partir du poids habituel à l’âge adulte ou du poids un an avant la date de référence, si cette donnée était disponible. Si cette variable n’était pas disponible, le poids au début de l’âge adulte a été utilisé comme indicateur. Le poids déclaré au moment du diagnostic ou lors de l’entretien dans les études cas-témoins n’a pas été utilisé afin d’éviter l’influence de la maladie sur le poids. Pour les deux études de cohorte prospectives (MCCS, UKBGS), nous avons utilisé le poids déclaré lors de l'entretien initial (avant le diagnostic)._

* _Les variables continues (âge des premières règles, AFTP, consommation d'alcool, taille et IMC) ont été modélisées à la fois comme des variables continues et catégorielles_

Source : [Associations conjointes d'un score de risque polygénique et de facteurs de risque environnementaux pour le cancer du sein dans le Breast Cancer Association Consortium](https://pmc.ncbi.nlm.nih.gov/articles/PMC5913605/#sec16)

# Current smoking

## Data import

In [8]:
import os
import pandas as pd
from google.cloud import bigquery

dataset = os.environ["WORKSPACE_CDR"]
client = bigquery.Client()

query_tobacco_exact = f"""
SELECT 
    obs.person_id,
    obs.observation_date AS tobacco_survey_date,
    obs.observation_source_concept_id AS tobacco_question_concept_id,
    obs.value_source_concept_id AS tobacco_answer_concept_id,
    c_src_r.concept_name AS tobacco_answer_label
FROM `{dataset}.observation` obs
LEFT JOIN `{dataset}.concept` c_src_r 
  ON obs.value_source_concept_id = c_src_r.concept_id

WHERE obs.observation_source_concept_id IN (
    1585857, -- Lifetime 100 cigarettes (Yes/No)
    1585860  -- Current smoking frequency (Every day / Some days / Not at all)
)
ORDER BY obs.person_id, tobacco_survey_date ASC
"""

df_tobacco_raw = client.query(query_tobacco_exact).to_dataframe()

print(f"Total raw response rows: {len(df_tobacco_raw)}")
print(f"Unique participants: {df_tobacco_raw['person_id'].nunique()}")

df_tobacco_raw.head()

Total raw response rows: 805135
Unique participants: 578451


,person_id,tobacco_survey_date,tobacco_question_concept_id,tobacco_answer_concept_id,tobacco_answer_label
0,1000000,2019-08-29,1585860,1585861,Smoke Frequency: Every Day
1,1000000,2019-08-29,1585857,1585858,100 Cigs Lifetime: Yes
2,1000004,2019-07-26,1585857,1585859,100 Cigs Lifetime: No
3,1000005,2018-06-19,1585857,1585859,100 Cigs Lifetime: No
4,1000012,2018-08-22,1585857,1585859,100 Cigs Lifetime: No


## Add column `smoking_status`

In [9]:
# 1. Isolateur de la question sur les 100 cigarettes (Concept 1585857)
df_100cigs = df_tobacco_raw[
    df_tobacco_raw["tobacco_question_concept_id"] == 1585857
].copy()

# 2. Conversion en variable binaire (1 = Oui / 0 = Non)
map_tabac_binaire = {
    1585858: 1,  # Oui (A fumé au moins 100 cigarettes)
    1585859: 0,  # Non (Moins de 100 cigarettes)
}

df_100cigs["smoking_status"] = df_100cigs["tobacco_answer_concept_id"].map(
    map_tabac_binaire
)

## Merge with breast cancer cohort

In [10]:
df_smoking = df.merge(df_100cigs, on='person_id', how='left')

In [11]:
df_smoking.drop(columns='tobacco_question_concept_id', inplace=True)

In [12]:
df_smoking

,person_id,inclusion_date,first_breast_cancer_date,delay_days,age_at_inclusion,has_bc,eur_rye,eas_rye,amr_rye,afr_rye,...,ancestry_80,biopsy_result,age_category,race_us_bcsc,race_us_bcsc_detail,breast_cancer_family_history_first_degree,tobacco_survey_date,tobacco_answer_concept_id,tobacco_answer_label,smoking_status
0,1700611,2019-09-17,NaN,NaN,70.255989,0,0.091868,0.000000,0.000000,0.848348,...,afr_rye,0,72,2,afr_rye,0,2019-09-17,1585858,100 Cigs Lifetime: Yes,1.0
1,1356439,2019-03-04,NaN,NaN,69.716632,0,0.065172,0.000000,0.025094,0.000000,...,sas_rye,0,67,3,eas_rye/sas_rye,0,2019-03-04,1585859,100 Cigs Lifetime: No,0.0
2,3451057,2023-01-18,NaN,NaN,71.594798,0,0.103268,0.000000,0.037591,0.000000,...,sas_rye,0,72,3,eas_rye/sas_rye,0,2023-01-18,1585859,100 Cigs Lifetime: No,0.0
3,1559161,2019-03-11,NaN,NaN,67.737166,0,0.006018,0.006429,0.965105,0.000000,...,amr_rye,0,67,8,amr_rye/mid_rye,0,2019-03-14,1585859,100 Cigs Lifetime: No,0.0
4,1468526,2019-10-07,NaN,NaN,63.310062,0,0.896282,0.000000,0.032661,0.000000,...,eur_rye,0,62,1,eur_rye,0,2019-10-07,1585858,100 Cigs Lifetime: Yes,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124896,4386725,2023-05-11,NaN,NaN,40.903491,0,0.894660,0.000000,0.045839,0.000000,...,eur_rye,1,42,1,eur_rye,0,2023-05-11,1585859,100 Cigs Lifetime: No,0.0
124897,2743914,2022-08-08,NaN,NaN,40.147844,0,0.980679,0.000000,0.000000,0.000000,...,eur_rye,0,42,1,eur_rye,0,2022-08-08,1585859,100 Cigs Lifetime: No,0.0
124898,1475516,2022-07-06,NaN,NaN,40.057495,0,0.848439,0.000000,0.049838,0.000000,...,eur_rye,0,42,1,eur_rye,0,2022-07-06,1585859,100 Cigs Lifetime: No,0.0
124899,9915233,2022-09-14,NaN,NaN,40.249144,0,0.673153,0.000000,0.050596,0.000000,...,eur / mid,0,42,1,eur_rye,0,2022-09-14,1585859,100 Cigs Lifetime: No,0.0


In [ ]:
df_smoking[['has_bc','smoking_status']].value_counts()

# Data's export

In [13]:
destination_filename = 'Datas/df_54_RiskFactors_CurrentSmoking.tsv'
df_smoking.to_csv(destination_filename, index=False)

# Récupère le nom du bucket Google Cloud depuis la variable d’environnement
my_bucket = os.getenv('WORKSPACE_BUCKET')

# Copie le fichier TSV local dans le dossier "Data" du bucket
args = ["gsutil", "cp", f"./{destination_filename}", f"{my_bucket}/Data/"]
output = subprocess.run(args, capture_output=True)

# Affiche les éventuelles erreurs retournées par gsutil
output.stderr

b'Copying file://./Datas/df_54_RiskFactors_CurrentSmoking.tsv [Content-Type=text/tab-separated-values]...\n/ [0 files][    0.0 B/ 23.2 MiB]                                                \r/ [1 files][ 23.2 MiB/ 23.2 MiB]                                                \r-\r\nOperation completed over 1 objects/23.2 MiB.                                     \n'